#  House Price Prediction Project
**Name:** Tirth Shah  
**Project:** Internship Week 1 — House Price Prediction  
**Date:** 21/05/2026

---

## TASK 1 — Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('Housing.csv')

# Display first 10 rows
print('First 10 rows:')
print(df.head(10))

# Dataset shape
print(f'\nShape: {df.shape[0]} rows × {df.shape[1]} columns')

# Target and Features
print('\nTarget: price')
print('Features:', list(df.columns[1:]))

# Missing values
print('\nMissing values:')
print(df.isnull().sum())

# Data types
print('\nData types:')
print(df.dtypes)

# Basic stats
print('\nBasic statistics:')
print(df.describe())

---

##  TASK 2 — Data Cleaning

Steps performed:
1. **Checked for duplicates** — Found 0 duplicate rows
2. **Checked for missing values** — Found 0 missing values
3. **Converted categorical columns to numeric:**
   - `mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, `prefarea` → mapped `yes/no` to `1/0`
   - `furnishingstatus` → one-hot encoding (semi-furnished, unfurnished)

In [ ]:
# Data Cleaning
df_clean = df.copy()

# Convert binary yes/no to 1/0
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'yes': 1, 'no': 0})

# One-hot encoding for furnishingstatus
df_clean = pd.get_dummies(df_clean, columns=['furnishingstatus'], drop_first=True)

# Convert booleans to integers
df_clean['furnishingstatus_semi-furnished'] = df_clean['furnishingstatus_semi-furnished'].astype(int)
df_clean['furnishingstatus_unfurnished'] = df_clean['furnishingstatus_unfurnished'].astype(int)

print('Cleaned dataset:')
print(df_clean.head())
print(f'\nShape: {df_clean.shape}')
print('\nAll numeric:', df_clean.dtypes.to_dict())

---

##  TASK 3 — Model Building

We trained two models:
1. **Linear Regression**
2. **Random Forest Regressor**

Split: 80% Training (436 samples) / 20% Testing (109 samples)

In [ ]:
# Define X and y
X = df_clean.drop('price', axis=1)
y = df_clean['price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print('=== LINEAR REGRESSION ===')
print(f'MAE:  ₹{lr_mae:,.0f}')
print(f'RMSE: ₹{lr_rmse:,.0f}')
print(f'R²:   {lr_r2:.4f}')

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print('\n=== RANDOM FOREST ===')
print(f'MAE:  ₹{rf_mae:,.0f}')
print(f'RMSE: ₹{rf_rmse:,.0f}')
print(f'R²:   {rf_r2:.4f}')

# Comparison
print('\n=== COMPARISON ===')
print(f'Linear Regression R²: {lr_r2:.4f}')
print(f'Random Forest R²:     {rf_r2:.4f}')
print('Winner: Linear Regression (better R² score!)')

---

##  TASK 4 — Visualization

### Chart 1: Histogram — Distribution of House Prices

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['price'], bins=40, color='steelblue', edgecolor='black', alpha=0.7)
plt.title('Distribution of House Prices', fontsize=16, fontweight='bold')
plt.xlabel('Price (₹)')
plt.ylabel('Frequency')
plt.ticklabel_format(style='plain', axis='x')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/chart1_price_distribution.png', dpi=150)
plt.show()

### Chart 2: Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt='.2f', 
           linewidths=0.5, mask=mask, cbar_kws={'shrink': .8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/chart2_correlation_heatmap.png', dpi=150)
plt.show()

### Chart 3: Actual vs Predicted Prices (Scatter Plot)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, lr_pred, alpha=0.6, color='darkgreen', edgecolors='black', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price (₹)', fontsize=12)
plt.ylabel('Predicted Price (₹)', fontsize=12)
plt.title('Actual vs Predicted House Prices (Linear Regression)', fontsize=16, fontweight='bold')
plt.legend()
plt.ticklabel_format(style='plain', axis='both')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('charts/chart3_actual_vs_predicted.png', dpi=150)
plt.show()

### Bonus Chart: Feature Importance (Random Forest)

In [ ]:
plt.figure(figsize=(10, 6))
fi = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=True)
plt.barh(fi['Feature'], fi['Importance'], color='teal', edgecolor='black')
plt.title('Feature Importance (Random Forest)', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('charts/chart4_feature_importance.png', dpi=150)
plt.show()

---

##  TASK 5 — Insights & Summary

### Key Findings:

**1. Which features influence house price the most?**
- **Area** is the strongest predictor (correlation = 0.54, highest RF importance ~48%). Bigger houses cost more.
- **Bathrooms** has a surprisingly high impact (coefficient = ₹1,094,445). Each additional bathroom adds significant value.
- **Air Conditioning** is highly valued (coefficient = ₹791,427). It's a premium feature buyers pay for.
- **Parking** and **Stories** also show strong positive correlations with price.
- **Unfurnished** status negatively impacts price (coefficient = -₹413,645).

**2. How accurate was the model?**
- The **Linear Regression model** achieved an **R² score of 0.6529**, meaning it explains about **65% of the price variation**.
- On average, predictions were off by about **₹9.7 lakhs** (MAE).
- Interestingly, Linear Regression outperformed Random Forest on this dataset, suggesting the relationships are fairly linear.

**3. What surprised me in the data?**
- **Bathrooms matter more than bedrooms!** The correlation and coefficient for bathrooms (0.52) is much higher than bedrooms (0.37). This suggests buyers prioritize quality/luxury over just room count.
- **Hot water heating** has a high coefficient (₹684,650) despite being rare in the dataset — indicating it's a premium feature.
- The price distribution is **right-skewed** — most houses are in the ₹3-5 crore range, with a few luxury properties going above ₹1 crore.

**4. One recommendation for a real estate business:**
> **Focus on 'Area + Bathrooms + AC' when pricing and marketing properties.** Renovating to add an extra bathroom or installing air conditioning can yield higher returns than simply adding bedrooms. Also, **avoid marketing unfurnished properties at premium rates** — they consistently sell for less. Prioritize properties with parking and main road access as these are strong positive price drivers.

---
